# Two Windows, One Throw

**Coaching question:** in the last second before the ball is thrown, how long did the offense have *at least one* of its two wide receivers in space?

Two receivers can each be open for the same amount of time, yet give the quarterback very different options:

- if they are open **at the same moments**, the windows stack, and the offense gets one short window;
- if they are open **at different moments**, the windows chain, and someone is open for longer.

This notebook measures that for every eligible play in Weeks 1–8 of the 2021 tracking data, picks two real plays that show the contrast, and writes `index.html`: two field replays on one shared slider with a timeline strip for each receiver.

**Metric — time with receiver space.** For each of the two selected wide receivers, take the distance to the closest defender in each of the 10 frames (1.0 s) immediately before `pass_forward`. A receiver is *in space* in a frame when that distance is **at least 2 yards**. Each frame is 0.1 s. We count frames where **either** receiver is in space and frames where **both** are.

Run the cells top to bottom from the project folder. The first run downloads about 870 MB of tracking data into `data/`.

## Setup

In [1]:
import json
import socket
import threading
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import IFrame, display

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
TRACKING_DIR = DATA_DIR / "tracking"
RAW_URL = "https://raw.githubusercontent.com/ThompsonJamesBliss/nfl-big-data-bowl-regional-event-data/main/data"
D3_SRC = "https://cdn.jsdelivr.net/npm/d3@7.9.0/dist/d3.min.js"

THRESHOLD_YD = 2.0               # in space when the closest defender is at least this far away
SENSITIVITY_YD = [1.5, 2.0, 2.5]  # thresholds for the sensitivity check
WINDOW_FRAMES = 10               # frames immediately before pass_forward
FPS = 10                         # tracking frames per second
FIELD_WIDTH = 160 / 3            # 53.3 yards
WEEKS = list(range(1, 9))        # all available weeks; use [1] for a quicker run

# Pair search: skip receivers who were never or always open (0.2-0.8 s only),
# and require at least a 0.3 s gap in "either" time between the two plays.
MIN_FRAMES_OPEN, MAX_FRAMES_OPEN = 2, 8
MIN_GAP_FRAMES = 3

KEYS = ["gameId", "playId"]

## 1. Data

Download the official files once. Later runs reuse the local copies.

In [2]:
def download(rel_path: str) -> Path:
    dest = DATA_DIR / rel_path
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_name(dest.name + ".part")
    urllib.request.urlretrieve(f"{RAW_URL}/{rel_path}", tmp)
    tmp.rename(dest)
    return dest


for name in ["games.csv", "plays.csv", "pffScoutingData.csv", "players.csv"]:
    download(name)

games = pd.read_csv(DATA_DIR / "games.csv")
game_ids = games.loc[games.week.isin(WEEKS), "gameId"].tolist()
with ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(lambda g: download(f"tracking/tracking_{g}.csv"), game_ids))

plays = pd.read_csv(DATA_DIR / "plays.csv")
scout = pd.read_csv(DATA_DIR / "pffScoutingData.csv")
players = pd.read_csv(DATA_DIR / "players.csv")
print(f"{len(game_ids)} games (weeks {WEEKS[0]}-{WEEKS[-1]}), {plays.gameId.isin(game_ids).sum():,} plays")

122 games (weeks 1-8), 8,557 plays


## 2. Check the notebook → D3 path

A single D3 dot, written to `checks/dot_test.html` and previewed from a small local web server. Open the same URL in a browser to confirm the page works outside Jupyter.

In [3]:
def start_server(directory: Path) -> int:
    """Serve the project folder on a free localhost port (once per kernel)."""
    global _server
    if "_server" in globals():
        return _server.server_address[1]
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        port = s.getsockname()[1]
    handler = partial(SimpleHTTPRequestHandler, directory=str(directory))
    handler.log_message = lambda *args: None
    _server = ThreadingHTTPServer(("127.0.0.1", port), handler)
    threading.Thread(target=_server.serve_forever, daemon=True).start()
    return port


PORT = start_server(ROOT)
(ROOT / "checks").mkdir(exist_ok=True)
(ROOT / "checks" / "dot_test.html").write_text(f"""<!doctype html>
<html><head><meta charset="utf-8"><script src="{D3_SRC}"></script></head>
<body style="margin:0;font-family:system-ui,sans-serif;background:#fff">
<svg id="s" width="220" height="80"></svg>
<script>
  const s = d3.select("#s");
  s.append("circle").attr("cx", 110).attr("cy", 34).attr("r", 14).attr("fill", "#d9682b");
  s.append("text").attr("x", 110).attr("y", 72).attr("text-anchor", "middle")
   .attr("font-size", 12).text("D3 " + d3.version + " is working");
</script></body></html>""")
print(f"http://127.0.0.1:{PORT}/checks/dot_test.html")
display(IFrame(f"http://127.0.0.1:{PORT}/checks/dot_test.html", width=240, height=90))

http://127.0.0.1:64418/checks/dot_test.html


## 3. Metric

Three small functions do all the measuring:

- `release_frame` finds the one `pass_forward` frame. The tag repeats on every player's row in that frame, so rows collapse to unique frames first. Plays tagged in more than one frame are dropped.
- `receiver_separation` returns, for each receiver and each of the 10 window frames, the distance to the closest defender. If the receiver or any expected defender is missing in a frame, that frame's distance is missing.
- `window_summary` turns two 10-frame distance sequences into seconds in space. It returns `None` (window unavailable) if either sequence is incomplete.

In [4]:
def release_frame(rows: pd.DataFrame):
    """The single pass_forward frame of a play, or None if there are zero or several."""
    frames = rows.loc[rows["event"] == "pass_forward", "frameId"].unique()
    return int(frames[0]) if len(frames) == 1 else None


def receiver_separation(win: pd.DataFrame, receivers: pd.DataFrame, expected_def: pd.DataFrame) -> pd.DataFrame:
    """Closest-defender distance for each receiver in each window step.

    win          tracking rows with gameId, playId, frameId, nflId, team, x, y, possessionTeam, rel
    receivers    gameId, playId, nflId of the selected receivers
    expected_def gameId, playId, n_def_expected (defenders listed by PFF for the play)
    """
    defense = win[(win.team != win.possessionTeam) & (win.team != "football")]

    # Full grid of play x receiver x step, so a missing row becomes a missing distance.
    grid = win[KEYS + ["rel"]].drop_duplicates().merge(
        pd.DataFrame({"step": range(WINDOW_FRAMES)}), how="cross")
    grid["frameId"] = grid.rel - WINDOW_FRAMES + grid.step
    grid = grid.merge(receivers[KEYS + ["nflId"]], on=KEYS)
    rx = grid.merge(win[KEYS + ["frameId", "nflId", "x", "y"]], on=KEYS + ["frameId", "nflId"], how="left")

    d = defense[KEYS + ["frameId", "nflId", "x", "y"]].rename(columns={"nflId": "defId", "x": "dx", "y": "dy"})
    pairs = rx.merge(d, on=KEYS + ["frameId"], how="left")
    pairs["sep"] = np.hypot(pairs.x - pairs.dx, pairs.y - pairs.dy)
    nearest = (pairs.sort_values("sep", na_position="last")
                    .drop_duplicates(KEYS + ["nflId", "step"]))

    seen = defense.groupby(KEYS + ["frameId"]).size().rename("n_def").reset_index()
    nearest = nearest.merge(seen, on=KEYS + ["frameId"], how="left").merge(expected_def, on=KEYS, how="left")
    complete = nearest.x.notna() & nearest.n_def.eq(nearest.n_def_expected)
    nearest.loc[~complete, ["sep", "defId"]] = np.nan
    return (nearest.sort_values(KEYS + ["nflId", "step"])
                   [KEYS + ["nflId", "step", "frameId", "sep", "defId"]]
                   .reset_index(drop=True))


def window_summary(sep_a, sep_b, threshold=THRESHOLD_YD):
    """Seconds in space for receiver a, receiver b, either, and both. None if a frame is missing."""
    sep_a, sep_b = np.asarray(sep_a, float), np.asarray(sep_b, float)
    if len(sep_a) != WINDOW_FRAMES or len(sep_b) != WINDOW_FRAMES:
        return None
    if np.isnan(sep_a).any() or np.isnan(sep_b).any():
        return None
    a, b = sep_a >= threshold, sep_b >= threshold
    secs = lambda mask: round(int(mask.sum()) / FPS, 1)
    return {"r1": secs(a), "r2": secs(b), "either": secs(a | b), "both": secs(a & b)}

### Sanity checks from the plan

In [5]:
OPEN, SHUT = 3.0, 1.0
four_early = [OPEN] * 4 + [SHUT] * 6
four_later = [SHUT] * 4 + [OPEN] * 4 + [SHUT] * 2

# Four open frames each, fully overlapping -> either 0.4 s, both 0.4 s
r = window_summary(four_early, four_early)
assert (r["either"], r["both"]) == (0.4, 0.4), r
# Four open frames each, fully staggered -> either 0.8 s, both 0.0 s
r = window_summary(four_early, four_later)
assert (r["either"], r["both"]) == (0.8, 0.0), r

# Toy play: receiver 10 is exactly 2 yd from defender 30; receiver 20 is 1 yd from defender 40.
toy = pd.DataFrame([
    dict(gameId=1, playId=1, frameId=f, nflId=n, team=t, x=x, y=y)
    for f in range(1, 11)
    for n, t, x, y in [(10, "OFF", 0.0, 0.0), (20, "OFF", 10.0, 0.0), (30, "DEF", 0.0, 2.0), (40, "DEF", 10.0, 1.0)]
]).assign(possessionTeam="OFF", rel=11)
toy_rx = pd.DataFrame({"gameId": 1, "playId": 1, "nflId": [10, 20]})
toy_def = pd.DataFrame({"gameId": [1], "playId": [1], "n_def_expected": [2]})
seq = lambda s, i: s.loc[s.nflId == i, "sep"].to_numpy()

# Separation of exactly 2 yards counts as in space
s = receiver_separation(toy, toy_rx, toy_def)
r = window_summary(seq(s, 10), seq(s, 20))
assert (r["r1"], r["r2"]) == (1.0, 0.0), r

# A missing defender or receiver observation makes the window unavailable
s = receiver_separation(toy[~((toy.nflId == 40) & (toy.frameId == 5))], toy_rx, toy_def)
assert window_summary(seq(s, 10), seq(s, 20)) is None
s = receiver_separation(toy[~((toy.nflId == 20) & (toy.frameId == 3))], toy_rx, toy_def)
assert window_summary(seq(s, 10), seq(s, 20)) is None

# The same release tag on several rows of one frame is one release frame
dup = pd.DataFrame({"frameId": [30, 30, 30, 12], "event": ["pass_forward"] * 3 + ["ball_snap"]})
assert release_frame(dup) == 30
assert release_frame(pd.DataFrame({"frameId": [30, 31], "event": ["pass_forward"] * 2})) is None

print("All metric sanity checks passed.")

All metric sanity checks passed.


## 4. Screen every eligible play

Join plays on `gameId` + `playId` and scouting on those keys plus `nflId`. Keep plays with exactly two wide receivers (by `officialPosition`) running routes, a single `pass_forward` frame, at least 10 tracked frames before it, and complete receiver and defender positions in those frames.

In [6]:
plays_w = plays[plays.gameId.isin(game_ids)]
roles = scout.merge(players[["nflId", "officialPosition", "displayName"]], on="nflId", how="left")
wr_routes = roles[(roles.pff_role == "Pass Route") & (roles.officialPosition == "WR")]
n_wr = wr_routes.groupby(KEYS).size()
two_wr = n_wr[n_wr == 2].reset_index()[KEYS]
receivers = wr_routes.merge(two_wr, on=KEYS)[KEYS + ["nflId", "displayName", "pff_positionLinedUp"]]
expected_def = (roles[roles.pff_role.isin(["Coverage", "Pass Rush"])]
                .groupby(KEYS).size().rename("n_def_expected").reset_index())
candidates = plays_w.merge(two_wr, on=KEYS)[KEYS + ["possessionTeam"]]

TRACK_COLS = ["gameId", "playId", "nflId", "frameId", "team", "x", "y", "event"]


def load_windows(game_id: int):
    """Window rows and release frames for one game's candidate plays."""
    t = pd.read_csv(TRACKING_DIR / f"tracking_{game_id}.csv", usecols=TRACK_COLS)
    t = t.merge(candidates[candidates.gameId == game_id], on=KEYS)
    tagged = t.loc[t.event == "pass_forward", KEYS + ["frameId", "event"]]
    rel = {k: release_frame(g) for k, g in tagged.groupby(KEYS)}
    rel = pd.DataFrame([(g, p, f) for (g, p), f in rel.items() if f is not None], columns=KEYS + ["rel"])
    t = t.merge(rel, on=KEYS)
    return t[(t.frameId >= t.rel - WINDOW_FRAMES) & (t.frameId < t.rel)], rel


with ThreadPoolExecutor(max_workers=4) as pool:
    loaded = list(pool.map(load_windows, game_ids))
windows = pd.concat([w for w, _ in loaded], ignore_index=True)
releases = pd.concat([r for _, r in loaded], ignore_index=True)
long_enough = releases[releases.rel > WINDOW_FRAMES]
windows = windows.merge(long_enough[KEYS], on=KEYS)

sep = receiver_separation(windows, receivers, expected_def)
print(f"{len(windows):,} tracking rows in the pre-release windows; {len(sep):,} receiver-frames measured")

411,240 tracking rows in the pre-release windows; 35,760 receiver-frames measured


127.0.0.1 - - [10/Nov/2026 13:02:14] "GET /checks/dot_test.html HTTP/1.1" 200 -


In [7]:
rows = []
for (g, p), d in sep.groupby(KEYS):
    ids = sorted(d.nflId.unique())
    seqs = [d.loc[d.nflId == i, "sep"].to_numpy() for i in ids]
    for th in SENSITIVITY_YD:
        s = window_summary(*seqs, threshold=th)
        rows.append({"gameId": g, "playId": p, "threshold": th, "rec1": ids[0], "rec2": ids[1],
                     "available": s is not None, **(s or {})})
summary = pd.DataFrame(rows).merge(
    plays[KEYS + ["possessionTeam", "defensiveTeam", "pff_passCoverage", "pff_passCoverageType", "passResult"]],
    on=KEYS)

main = summary[(summary.threshold == THRESHOLD_YD) & summary.available].copy()
frames = lambda s: (s * FPS).round().astype(int)
main["hi"] = frames(main[["r1", "r2"]].max(axis=1))
main["lo"] = frames(main[["r1", "r2"]].min(axis=1))
main["either_f"] = frames(main.either)
pair_pool = main[(main.lo >= MIN_FRAMES_OPEN) & (main.hi <= MAX_FRAMES_OPEN)]

funnel = pd.DataFrame([
    ("Pass plays, weeks %d-%d" % (WEEKS[0], WEEKS[-1]), len(plays_w)),
    ("Exactly two WR route runners", len(candidates)),
    ("Single pass_forward frame", len(releases)),
    ("At least 10 frames before release", len(long_enough)),
    ("Complete receiver + defender positions (eligible)", len(main)),
    ("Each WR in space 0.2-0.8 s (pair pool)", len(pair_pool)),
], columns=["step", "plays"])
funnel

,step,plays
0,"Pass plays, weeks 1-8",8557
1,Exactly two WR route runners,2034
2,Single pass_forward frame,1788
3,At least 10 frames before release,1788
4,Complete receiver + defender positions (eligible),1788
5,Each WR in space 0.2-0.8 s (pair pool),99


### What the full screen shows

How often do the two receivers' windows chain together instead of stacking? "Added time" is the either-time minus the better single receiver's time. It is zero when one receiver's window contains the other's. Chaining is only possible when both receivers get some space and neither is open for the whole second, so the share is reported for those plays.

In [8]:
added = main.either - main[["r1", "r2"]].max(axis=1)
# Chaining is only possible when both receivers get some space and neither is open the whole second.
could_chain = (main.lo >= 1) & (main.hi <= WINDOW_FRAMES - 1)
context = {
    "eligible": len(main),
    "could_chain": int(could_chain.sum()),
    "chained": int((added > 0).sum()),
    "chained_share": float((added[could_chain] > 0).mean()),
    "mean_added_when_chained": float(added[added > 0].mean()),
}
print(f"Eligible plays: {context['eligible']:,}")
print(f"Plays where both WRs had some space and neither was open the whole second: {context['could_chain']:,}")
print(f"  windows chained (either > better single receiver) on {context['chained']:,} "
      f"({context['chained_share']:.0%}), adding {context['mean_added_when_chained']:.2f} s on average")
pd.crosstab(frames(main.either), frames(main.both), margins=True).rename_axis(
    index="either (frames)", columns="both (frames)")

Eligible plays: 1,788
Plays where both WRs had some space and neither was open the whole second: 166
  windows chained (either > better single receiver) on 83 (50%), adding 0.19 s on average


both (frames),0,1,2,3,4,5,6,7,8,9,10,All
either (frames),,,,,,,,,,,,
0,143,0,0,0,0,0,0,0,0,0,0,143
1,28,0,0,0,0,0,0,0,0,0,0,28
2,28,3,0,0,0,0,0,0,0,0,0,31
3,31,1,0,1,0,0,0,0,0,0,0,33
4,22,3,3,1,1,0,0,0,0,0,0,30
5,26,1,3,1,2,0,0,0,0,0,0,33
6,25,3,4,1,4,2,1,0,0,0,0,40
7,28,1,3,3,4,1,0,1,0,0,0,41
8,29,2,2,6,5,1,2,2,1,0,0,50


## 5. Pick the pair

Group the pair pool by coverage and by each receiver's individual time in space (the sorted pair of frame counts). Within each group, every play gives its receivers the same individual time. The best group is the one with the widest spread in *either* time. Ties go to more evenly matched receivers, then to the larger group. From the winning group, take the play with the least *either* time (windows stacked) and the play with the most (windows chained). If no same-coverage group reaches a 0.3 s gap, coverage is dropped from the grouping.

In [9]:
def rank_groups(pool: pd.DataFrame, by: list) -> pd.DataFrame:
    g = (pool.groupby(by)
             .agg(n=("playId", "size"), either_min=("either_f", "min"), either_max=("either_f", "max"))
             .reset_index())
    g = g[g.n >= 2].assign(gap=lambda x: x.either_max - x.either_min, imbalance=lambda x: x.hi - x.lo)
    return g.sort_values(["gap", "imbalance", "n"], ascending=[False, True, False]).reset_index(drop=True)


by = ["pff_passCoverage", "hi", "lo"]
groups = rank_groups(pair_pool, by)
same_coverage = len(groups) > 0 and groups.gap.iloc[0] >= MIN_GAP_FRAMES
if not same_coverage:
    by = ["hi", "lo"]
    groups = rank_groups(pair_pool, by)
best = groups.iloc[0]
members = pair_pool[np.logical_and.reduce([pair_pool[c] == best[c] for c in by])]
stacked = members.sort_values(["either_f", "gameId", "playId"]).iloc[0]
chained = members.sort_values(["either_f", "gameId", "playId"], ascending=[False, True, True]).iloc[0]
assert chained.either > stacked.either

print("Matched on coverage:" if same_coverage else "No same-coverage pair; matched on receiver time only:")
display(groups.head(8))
pair = pd.DataFrame([stacked, chained], index=["stacked", "chained"])
pair[KEYS + ["possessionTeam", "defensiveTeam", "pff_passCoverage", "r1", "r2", "either", "both", "passResult"]]

Matched on coverage:


,pff_passCoverage,hi,lo,n,either_min,either_max,gap,imbalance
0,Cover-3,6,6,2,6,10,4,0
1,Cover-3,5,4,3,5,9,4,1
2,Cover-2,5,4,2,5,9,4,1
3,Cover-1,7,5,3,7,10,3,2
4,Cover-3,7,4,3,7,10,3,3
5,Cover-1,7,3,2,7,10,3,4
6,Quarters,8,7,3,8,10,2,1
7,Cover-1,8,6,2,8,10,2,2


,gameId,playId,possessionTeam,defensiveTeam,pff_passCoverage,r1,r2,either,both,passResult
stacked,2021091910,1621,ATL,TB,Cover-3,0.6,0.6,0.6,0.6,C
chained,2021101709,2778,DEN,LV,Cover-3,0.6,0.6,1.0,0.2,C


### Threshold sensitivity

The same two plays measured at 1.5, 2.0, and 2.5 yards.

In [10]:
sens = (summary.merge(pair[KEYS].assign(play=pair.index), on=KEYS)
               .set_index(["play", "threshold"])[["r1", "r2", "either", "both"]]
               .unstack("play").swaplevel(axis=1)
               .reindex(columns=pd.MultiIndex.from_product([["stacked", "chained"], ["r1", "r2", "either", "both"]])))
sens

stacked                  chained                 
               r1   r2 either both      r1   r2 either both
threshold                                                  
1.5           0.8  0.9    0.9  0.8     1.0  0.9    1.0  0.9
2.0           0.6  0.6    0.6  0.6     0.6  0.6    1.0  0.2
2.5           0.4  0.4    0.4  0.4     0.5  0.3    0.8  0.0

## 6. Build the replay

Reload the two selected plays and flip left-moving plays so the offense always moves up the screen. Positions are in yards from the line of scrimmage. The data embedded in `index.html` includes every player in the 10 window frames, the two receivers' routes from the snap, and the measured distances.

In [11]:
RESULT = {"C": "Complete", "I": "Incomplete", "IN": "Interception", "S": "Sack", "R": "Scramble"}
r1 = lambda v: round(float(v), 2)


def play_payload(row: pd.Series, label: str, tag: str) -> dict:
    g, p = int(row.gameId), int(row.playId)
    info = plays[(plays.gameId == g) & (plays.playId == p)].iloc[0]
    game = games[games.gameId == g].iloc[0]
    t = pd.read_csv(TRACKING_DIR / f"tracking_{g}.csv")
    t = t[t.playId == p].copy()

    left = t.playDirection.iloc[0] == "left"
    if left:
        t["x"], t["y"] = 120 - t.x, FIELD_WIDTH - t.y
    los = 120 - info.absoluteYardlineNumber if left else info.absoluteYardlineNumber
    t["down_field"] = t.x - los

    rel = release_frame(t)
    snaps = t.loc[t.event == "ball_snap", "frameId"].unique()
    snap = int(snaps[0]) if len(snaps) == 1 else int(t.frameId.min())
    ball_at_snap = t[(t.team == "football") & (t.frameId == snap)].iloc[0]
    assert abs(ball_at_snap.down_field) < 1.5, "line of scrimmage does not match the ball at the snap"

    s = sep[(sep.gameId == g) & (sep.playId == p)]
    recs = []
    for i in [int(row.rec1), int(row.rec2)]:
        rs = s[s.nflId == i].sort_values("step")
        who = roles[(roles.gameId == g) & (roles.playId == p) & (roles.nflId == i)].iloc[0]
        tr = t[(t.nflId == i) & t.frameId.between(snap, rel - 1)].sort_values("frameId")
        recs.append({
            "id": i,
            "name": who.displayName,
            "jersey": int(tr.jerseyNumber.iloc[0]),
            "aligned": who.pff_positionLinedUp,
            "sep": [r1(v) for v in rs.sep],
            "near": [int(v) for v in rs.defId],
            "open": [bool(v >= THRESHOLD_YD) for v in rs.sep],
            "trail": [[r1((f - rel) / FPS), r1(x), r1(y)] for f, x, y in zip(tr.frameId, tr.down_field, tr.y)],
        })
    for rec in recs:
        rec["seconds"] = round(sum(rec["open"]) / FPS, 1)

    frames_out = []
    for step in range(WINDOW_FRAMES):
        f = t[t.frameId == rel - WINDOW_FRAMES + step]
        ppl = f[f.team != "football"]
        ball = f[f.team == "football"].iloc[0]
        frames_out.append({
            "players": [[int(n), "O" if tm == info.possessionTeam else "D", r1(x), r1(y)]
                        for n, tm, x, y in zip(ppl.nflId, ppl.team, ppl.down_field, ppl.y)],
            "ball": [r1(ball.down_field), r1(ball.y)],
        })

    either = [a or b for a, b in zip(recs[0]["open"], recs[1]["open"])]
    both = [a and b for a, b in zip(recs[0]["open"], recs[1]["open"])]
    return {
        "key": tag, "label": label,
        "gameId": g, "playId": p, "week": int(game.week),
        "offense": info.possessionTeam, "defense": info.defensiveTeam,
        "quarter": int(info.quarter), "clock": info.gameClock,
        "down": int(info.down), "yardsToGo": int(info.yardsToGo),
        "coverage": info.pff_passCoverage, "coverageType": info.pff_passCoverageType,
        "formation": info.offenseFormation, "result": RESULT.get(info.passResult, info.passResult),
        "description": info.playDescription,
        "snapToRelease": round((rel - snap) / FPS, 1),
        "receivers": recs,
        "either": round(sum(either) / FPS, 1), "both": round(sum(both) / FPS, 1),
        "eitherSeq": either,
        "frames": frames_out,
    }


payload_plays = [play_payload(stacked, "Windows stacked", "A"),
                 play_payload(chained, "Windows chained", "B")]

# One downfield range for both fields, so they share a scale.
xs = [pl[2] for pp in payload_plays for fr in pp["frames"] for pl in fr["players"]]
xs += [q[1] for pp in payload_plays for rec in pp["receivers"] for q in rec["trail"]]
x_domain = [float(np.floor(min(xs)) - 2), float(np.ceil(max(xs)) + 2)]

for pp, row in zip(payload_plays, [stacked, chained]):
    assert pp["either"] == row.either and pp["both"] == row.both  # replay matches the screen

sens_rows = (summary.merge(pair[KEYS].assign(play=["A", "B"]), on=KEYS)
                    .sort_values(["threshold", "play"]))
A, B = payload_plays
indiv = A["receivers"][0]["seconds"]
same_indiv = all(r["seconds"] == indiv for pp in payload_plays for r in pp["receivers"])
headline = (
    f"In both {A['coverage']} plays, each wide receiver was in space for "
    + (f"{indiv:.1f} s" if same_indiv else "a similar amount of time")
    + f" in the last second before the throw. In play A ({A['offense']} vs {A['defense']}) they were open at the same moments, "
    f"so the offense had {A['either']:.1f} s with at least one receiver in space. "
    f"In play B ({B['offense']} vs {B['defense']}) they took turns, so it had {B['either']:.1f} s."
)

DATA = {
    "threshold": THRESHOLD_YD, "fps": FPS, "window": WINDOW_FRAMES, "fieldWidth": round(FIELD_WIDTH, 3),
    "times": [round((s - WINDOW_FRAMES) / FPS, 1) for s in range(WINDOW_FRAMES)],
    "xDomain": x_domain,
    "headline": headline,
    "plays": payload_plays,
    "funnel": funnel.values.tolist(),
    "context": {k: (round(v, 3) if isinstance(v, float) else v) for k, v in context.items()},
    "sensitivity": [{"threshold": float(r.threshold), "play": r.play, "r1": float(r.r1), "r2": float(r.r2),
                     "either": float(r.either), "both": float(r.both)} for r in sens_rows.itertuples()],
}
print(headline)
print(f"Payload: {len(json.dumps(DATA)) / 1024:.0f} KB")

In both Cover-3 plays, each wide receiver was in space for 0.6 s in the last second before the throw. In play A (ATL vs TB) they were open at the same moments, so the offense had 0.6 s with at least one receiver in space. In play B (DEN vs LV) they took turns, so it had 1.0 s.
Payload: 20 KB


### Page template

Plain HTML, CSS, and D3 v7. The play data goes in as one JSON block.

In [12]:
TEMPLATE = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Two Windows, One Throw</title>
<meta name="description" content="Two real NFL plays where receivers had the same time in space, at different moments.">
<script src="__D3__"></script>
<style>
  :root {
    --bg: #f6f5f1; --card: #ffffff; --ink: #1c2330; --muted: #5d6879; --line: #dcdfe4;
    --turf: #eef3ec; --turf-line: #cad7c6; --los: #3b5bdb; --first: #e0b100;
    --closed: #e7e9ed; --either: #39424f;
  }
  * { box-sizing: border-box; }
  body { margin: 0; background: var(--bg); color: var(--ink);
         font: 15px/1.5 system-ui, -apple-system, "Segoe UI", Roboto, sans-serif; }
  main { max-width: 1180px; margin: 0 auto; padding: 28px 16px 56px; }
  h1 { font-size: 30px; line-height: 1.15; margin: 0 0 8px; letter-spacing: -0.01em; }
  .lede { color: var(--muted); max-width: 780px; margin: 0 0 6px; }
  .headline { max-width: 780px; margin: 10px 0 18px; font-size: 16px; }
  .controls { position: sticky; top: 0; z-index: 5; display: flex; gap: 14px; align-items: center;
              background: var(--bg); padding: 10px 0; border-bottom: 1px solid var(--line); margin-bottom: 18px; }
  button { font: inherit; font-weight: 600; padding: 6px 16px; border-radius: 6px; border: 1px solid var(--ink);
           background: var(--ink); color: #fff; cursor: pointer; min-width: 88px; }
  button:focus-visible, input:focus-visible { outline: 2px solid var(--los); outline-offset: 2px; }
  input[type=range] { flex: 1; min-width: 160px; accent-color: var(--ink); }
  #time { font-variant-numeric: tabular-nums; font-weight: 700; min-width: 170px; text-align: right; }
  .legend { display: flex; flex-wrap: wrap; gap: 6px 18px; font-size: 13px; color: var(--muted); margin: -6px 0 16px; }
  .legend span { display: inline-flex; align-items: center; gap: 6px; }
  .legend i { display: inline-block; width: 11px; height: 11px; border-radius: 50%; }
  .plays { display: grid; grid-template-columns: repeat(auto-fit, minmax(min(100%, 500px), 1fr)); gap: 20px; }
  .card { background: var(--card); border: 1px solid var(--line); border-radius: 10px; padding: 16px 16px 12px; min-width: 0; }
  .tag { display: inline-block; font-size: 12px; font-weight: 700; letter-spacing: .05em; text-transform: uppercase;
         color: var(--muted); }
  .card h2 { font-size: 19px; margin: 2px 0 0; }
  .meta { color: var(--muted); font-size: 13px; margin: 2px 0 10px; }
  .totals { display: flex; gap: 22px; flex-wrap: wrap; margin: 6px 0 12px; }
  .stat .v { font-size: 28px; font-weight: 750; font-variant-numeric: tabular-nums; line-height: 1.1; }
  .stat .k { font-size: 12px; color: var(--muted); }
  .stat.main .v { color: var(--ink); }
  svg { display: block; width: 100%; height: auto; }
  .desc { font-size: 13px; color: var(--muted); margin: 8px 0 0; }
  .yard-label { font-size: 10px; fill: #7d8a78; }
  .rec-label { font-size: 11px; font-weight: 700; paint-order: stroke; stroke: #fff; stroke-width: 3px; }
  .sep-label { font-size: 10px; font-weight: 700; paint-order: stroke; stroke: #fff; stroke-width: 3px; }
  .strip-label { font-size: 12px; fill: var(--ink); }
  .strip-sub { font-size: 10px; fill: var(--muted); }
  .tick { font-size: 10px; fill: var(--muted); font-variant-numeric: tabular-nums; }
  section.method { margin-top: 32px; display: grid; grid-template-columns: repeat(auto-fit, minmax(min(100%, 340px), 1fr)); gap: 20px; }
  section.method > div { background: var(--card); border: 1px solid var(--line); border-radius: 10px; padding: 16px; }
  section.method h3 { font-size: 15px; margin: 0 0 8px; }
  section.method p, section.method li { font-size: 14px; color: var(--muted); }
  table { border-collapse: collapse; width: 100%; font-size: 13px; font-variant-numeric: tabular-nums; }
  th, td { text-align: right; padding: 4px 6px; border-bottom: 1px solid var(--line); }
  th:first-child, td:first-child { text-align: left; }
  th { color: var(--muted); font-weight: 600; }
  @media (max-width: 620px) {
    h1 { font-size: 24px; }
    .controls { flex-wrap: wrap; }
    #time { text-align: left; min-width: 0; }
  }
</style>
</head>
<body>
<main>
  <h1>Two Windows, One Throw</h1>
  <p class="lede">How much time did the offense have with at least one of its two wide receivers in space before the throw? Two real 2021 plays, measured over the last second before release.</p>
  <p class="headline" id="headline"></p>
  <div class="controls">
    <button id="play" type="button">Play</button>
    <input id="slider" type="range" min="0" max="9" step="1" value="0" aria-label="Time before release">
    <span id="time" aria-live="polite"></span>
  </div>
  <div class="legend" id="legend"></div>
  <div class="plays" id="plays"></div>
  <section class="method">
    <div>
      <h3>How it is measured</h3>
      <p id="method"></p>
      <p>Passing lanes, pass rush pressure, and the quarterback's reads are outside the measurement. It describes the options the two receivers created, not where the ball went: in both plays shown, the pass went to a tight end.</p>
    </div>
    <div>
      <h3>Threshold sensitivity</h3>
      <table id="sens"></table>
    </div>
    <div>
      <h3>Plays screened</h3>
      <table id="funnel"></table>
      <p id="context"></p>
    </div>
  </section>
</main>
<script>
const D = __DATA__;
const N = D.window;
const COLORS = { r: ["#d9682b", "#2b7bb9"], O: "#8b97a8", D: "#c2453d", ball: "#7a4a22" };
const W = 500, PAD = 12;
const pxPerYd = (W - 2 * PAD) / D.fieldWidth;
const H = Math.round((D.xDomain[1] - D.xDomain[0]) * pxPerYd + 2 * PAD);
const sx = d3.scaleLinear([D.fieldWidth, 0], [PAD, W - PAD]);   // sideline to sideline (offense's left on the left)
const sy = d3.scaleLinear(D.xDomain, [H - PAD, PAD]);           // downfield is up
const ringR = D.threshold * pxPerYd;
const fmt = v => v.toFixed(1) + " s";
const timeText = t => (t < 0 ? "−" : "") + Math.abs(t).toFixed(1) + " s before release";

d3.select("#headline").text(D.headline);
d3.select("#method").text(
  `For each of the two wide receivers, the distance to the closest defender is measured in each of the ${N} frames ` +
  `(${N / D.fps} s) immediately before the pass. A receiver is in space when that distance is at least ${D.threshold} yards. ` +
  `Each frame counts 0.1 s. "Either" counts frames where at least one receiver is in space; "both" counts frames where both are.`);

const legend = d3.select("#legend");
[["Receiver 1", COLORS.r[0]], ["Receiver 2", COLORS.r[1]], ["Other offense", COLORS.O], ["Defense", COLORS.D]]
  .forEach(([k, c]) => legend.append("span").html(`<i style="background:${c}"></i>${k}`));
legend.append("span").html(`<i style="background:transparent;border:1.5px dashed #6b7280"></i>${D.threshold}-yard ring (filled = in space)`);

const cards = d3.select("#plays").selectAll("article").data(D.plays).join("article").attr("class", "card");
cards.each(function (p) { buildCard(d3.select(this), p); });

function buildCard(card, p) {
  card.append("div").attr("class", "tag").text(`Play ${p.key} · ${p.label}`);
  card.append("h2").text(p.receivers.map(r => r.name).join(" & "));
  card.append("div").attr("class", "meta").text(
    `${p.offense} vs ${p.defense} · Week ${p.week} · Q${p.quarter} ${p.clock} · ` +
    `${p.down}&${p.yardsToGo} · ${p.coverage} (${p.coverageType}) · ${p.result}`);

  const totals = card.append("div").attr("class", "totals");
  const stat = (v, k, cls) => { const s = totals.append("div").attr("class", "stat " + (cls || "")); s.append("div").attr("class", "v").text(v); s.append("div").attr("class", "k").text(k); };
  stat(fmt(p.either), "either receiver in space", "main");
  stat(fmt(p.both), "both in space");
  p.receivers.forEach((r, i) => stat(fmt(r.seconds), r.name.split(" ").slice(-1)[0] + " alone"));
  totals.selectAll(".stat").filter((d, i) => i >= 2).select(".v").style("color", (d, i) => COLORS.r[i]);

  // ---- Field
  const svg = card.append("svg").attr("viewBox", `0 0 ${W} ${H}`)
    .attr("role", "img").attr("aria-label", `Field replay for play ${p.key}`);
  svg.append("rect").attr("x", PAD).attr("y", PAD).attr("width", W - 2 * PAD).attr("height", H - 2 * PAD)
    .attr("fill", "var(--turf)").attr("stroke", "var(--turf-line)");
  const lines = d3.range(Math.ceil(D.xDomain[0] / 5) * 5, D.xDomain[1], 5);
  svg.append("g").selectAll("line").data(lines).join("line")
    .attr("x1", PAD).attr("x2", W - PAD).attr("y1", sy).attr("y2", sy)
    .attr("stroke", d => d === 0 ? "var(--los)" : "var(--turf-line)").attr("stroke-width", d => d === 0 ? 2 : 1);
  svg.append("g").selectAll("text").data(lines.filter(d => d !== 0)).join("text").attr("class", "yard-label")
    .attr("x", PAD + 4).attr("y", d => sy(d) - 3).text(d => (d > 0 ? "+" : "") + d);
  svg.append("text").attr("class", "yard-label").attr("x", PAD + 4).attr("y", sy(0) - 3)
    .attr("fill", "var(--los)").style("fill", "var(--los)").text("line of scrimmage");
  if (p.yardsToGo < D.xDomain[1]) {
    svg.append("line").attr("x1", PAD).attr("x2", W - PAD).attr("y1", sy(p.yardsToGo)).attr("y2", sy(p.yardsToGo))
      .attr("stroke", "var(--first)").attr("stroke-width", 2).attr("opacity", 0.8);
  }

  const gTrail = svg.append("g"), gRing = svg.append("g"), gLink = svg.append("g"),
        gPlayers = svg.append("g"), gBall = svg.append("g"), gLabels = svg.append("g");
  const line = d3.line().x(d => sx(d[2])).y(d => sy(d[1])).curve(d3.curveCatmullRom);

  // ---- Strips: one row per receiver plus an "either" row
  const LABEL = 128, CELL = (W - LABEL - 4) / N, ROW = 22, GAP = 7;
  const rows = [
    ...p.receivers.map((r, i) => ({ label: r.name, sub: `#${r.jersey} ${r.aligned}`, cells: r.open, color: COLORS.r[i] })),
    { label: "Either in space", sub: "at least one receiver", cells: p.eitherSeq, color: "var(--either)" },
  ];
  const SH = rows.length * (ROW + GAP) + 18;
  const strips = card.append("svg").attr("viewBox", `0 0 ${W} ${SH}`).style("margin-top", "10px")
    .attr("role", "img").attr("aria-label", `Time in space by receiver for play ${p.key}`);
  rows.forEach((row, ri) => {
    const y = ri * (ROW + GAP);
    if (ri === 2) strips.append("line").attr("x1", 0).attr("x2", W).attr("y1", y - GAP / 2).attr("y2", y - GAP / 2).attr("stroke", "var(--line)");
    strips.append("text").attr("class", "strip-label").attr("x", 0).attr("y", y + 10).text(row.label);
    strips.append("text").attr("class", "strip-sub").attr("x", 0).attr("y", y + 21).text(row.sub);
    strips.append("g").selectAll("rect").data(row.cells).join("rect")
      .attr("x", (d, i) => LABEL + i * CELL + 1).attr("y", y).attr("width", CELL - 2).attr("height", ROW).attr("rx", 3)
      .style("fill", d => d ? row.color : "var(--closed)");
  });
  strips.append("g").selectAll("text").data(D.times).join("text").attr("class", "tick")
    .attr("x", (d, i) => LABEL + i * CELL + CELL / 2).attr("y", SH - 4).attr("text-anchor", "middle")
    .text(d => d.toFixed(1));
  const marker = strips.append("rect").attr("y", -3).attr("height", rows.length * (ROW + GAP) - GAP + 6)
    .attr("width", CELL + 2).attr("rx", 4).attr("fill", "none").attr("stroke", "var(--ink)").attr("stroke-width", 2);

  card.append("p").attr("class", "desc").text(p.description);

  p.draw = (step, dur) => {
    const t = D.times[step];
    const fr = p.frames[step];
    const pos = new Map(fr.players.map(d => [d[0], d]));
    const recIds = new Set(p.receivers.map(r => r.id));
    const tr = sel => dur ? sel.transition().duration(dur).ease(d3.easeLinear) : sel;

    gTrail.selectAll("path").data(p.receivers).join("path")
      .attr("fill", "none").attr("stroke-width", 2).attr("stroke-linecap", "round").attr("opacity", 0.55)
      .attr("stroke", (r, i) => COLORS.r[i])
      .attr("d", r => line(r.trail.filter(q => q[0] <= t + 1e-6)));

    const recs = p.receivers.map((r, i) => ({ r, i, me: pos.get(r.id), def: pos.get(r.near[step]), sep: r.sep[step], open: r.open[step] }));

    tr(gRing.selectAll("circle").data(recs).join("circle")
      .attr("stroke-width", 1.5)
      .attr("stroke", d => d.open ? COLORS.r[d.i] : "#6b7280")
      .attr("stroke-dasharray", d => d.open ? null : "4 3")
      .attr("fill", d => COLORS.r[d.i]).attr("fill-opacity", d => d.open ? 0.18 : 0)
      .attr("r", ringR))
      .attr("cx", d => sx(d.me[3])).attr("cy", d => sy(d.me[2]));

    tr(gLink.selectAll("line").data(recs).join("line")
      .attr("stroke", d => d.open ? COLORS.r[d.i] : "#6b7280").attr("stroke-width", 1.5))
      .attr("x1", d => sx(d.me[3])).attr("y1", d => sy(d.me[2]))
      .attr("x2", d => sx(d.def[3])).attr("y2", d => sy(d.def[2]));

    tr(gPlayers.selectAll("circle").data(fr.players, d => d[0]).join("circle")
      .attr("r", d => recIds.has(d[0]) ? 7 : 5)
      .attr("fill", d => recIds.has(d[0]) ? COLORS.r[p.receivers.findIndex(r => r.id === d[0])] : COLORS[d[1]])
      .attr("stroke", "#fff").attr("stroke-width", 1.2))
      .attr("cx", d => sx(d[3])).attr("cy", d => sy(d[2]));

    tr(gBall.selectAll("ellipse").data([fr.ball]).join("ellipse")
      .attr("rx", 4).attr("ry", 2.6).attr("fill", COLORS.ball))
      .attr("cx", d => sx(d[1])).attr("cy", d => sy(d[0]));

    const labels = gLabels.selectAll("g.rl").data(recs).join(enter => {
      const g = enter.append("g").attr("class", "rl");
      g.append("text").attr("class", "rec-label").attr("text-anchor", "middle");
      g.append("text").attr("class", "sep-label").attr("text-anchor", "start");
      return g;
    });
    labels.select(".rec-label").style("fill", d => COLORS.r[d.i]).text(d => d.r.name.split(" ").slice(-1)[0]);
    labels.select(".sep-label").style("fill", d => d.open ? COLORS.r[d.i] : "#4b5563").text(d => d.sep.toFixed(1) + " yd");
    tr(labels.select(".rec-label")).attr("x", d => sx(d.me[3])).attr("y", d => sy(d.me[2]) - ringR - 4);
    tr(labels.select(".sep-label")).attr("x", d => sx(d.me[3]) + ringR + 4).attr("y", d => sy(d.me[2]) + 4);

    marker.attr("x", LABEL + step * CELL);
  };
}

// ---- Shared time control
const slider = d3.select("#slider").attr("max", N - 1);
let step = 0, timer = null;
function show(s, dur) {
  step = s;
  slider.property("value", s).attr("aria-valuetext", timeText(D.times[s]));
  d3.select("#time").text(timeText(D.times[s]));
  D.plays.forEach(p => p.draw(s, dur));
}
slider.on("input", e => { stop(); show(+e.target.value, 0); });
function stop() { if (timer) { timer.stop(); timer = null; } d3.select("#play").text("Play"); }
d3.select("#play").on("click", () => {
  if (timer) return stop();
  d3.select("#play").text("Pause");
  if (step === N - 1) show(0, 0);
  timer = d3.interval(() => {
    if (step === N - 1) return stop();
    show(step + 1, 380);
  }, 450);
});
show(0, 0);

// ---- Method tables
const sens = d3.select("#sens");
sens.append("tr").html("<th>Threshold</th><th>A either</th><th>A both</th><th>B either</th><th>B both</th>");
d3.groups(D.sensitivity, d => d.threshold).forEach(([th, rs]) => {
  const a = rs.find(r => r.play === "A"), b = rs.find(r => r.play === "B");
  sens.append("tr").html(`<td>${th.toFixed(1)} yd</td><td>${fmt(a.either)}</td><td>${fmt(a.both)}</td><td>${fmt(b.either)}</td><td>${fmt(b.both)}</td>`)
    .style("font-weight", th === D.threshold ? 700 : null);
});
const funnel = d3.select("#funnel");
D.funnel.forEach(([k, v]) => funnel.append("tr").html(`<td>${k}</td><td>${v.toLocaleString()}</td>`));
d3.select("#context").text(
  `On ${D.context.could_chain.toLocaleString()} eligible plays, both receivers had some space but neither was open the whole second. ` +
  `Their windows chained (someone open longer than the better receiver alone) on ${D.context.chained.toLocaleString()} of them ` +
  `(${Math.round(100 * D.context.chained_share)}%), adding ${D.context.mean_added_when_chained.toFixed(2)} s on average.`);
</script>
</body>
</html>
"""

html = TEMPLATE.replace("__D3__", D3_SRC).replace("__DATA__", json.dumps(DATA, separators=(",", ":")))
(ROOT / "index.html").write_text(html)
print(f"Wrote index.html ({len(html) / 1024:.0f} KB)")

Wrote index.html (33 KB)


## 7. Preview

The same page served locally. Open the printed URL in a browser to test it outside Jupyter.

In [13]:
url = f"http://127.0.0.1:{PORT}/index.html"
print(url)
display(IFrame(url, width="100%", height=1250))

http://127.0.0.1:64418/index.html


## 8. Findings

In [ ]:
A, B = DATA["plays"]
print(DATA["headline"])
print()
print(f"Eligible plays screened: {context['eligible']:,} (weeks {WEEKS[0]}-{WEEKS[-1]}).")
print(f"Where chaining was possible ({context['could_chain']} plays), windows chained {context['chained_share']:.0%} of the time, "
      f"adding {context['mean_added_when_chained']:.2f} s on average.")
print("Sensitivity (either time, A vs B): " + "; ".join(
    f"{s:.1f} yd: {a:.1f} s vs {b:.1f} s"
    for s, a, b in sens_rows.pivot(index="threshold", columns="play", values="either").itertuples()))

In both Cover-3 plays, each wide receiver was in space for 0.6 s in the last second before the throw. In play A (ATL vs TB) they were open at the same moments, so the offense had 0.6 s with at least one receiver in space. In play B (DEN vs LV) they took turns, so it had 1.0 s.

Eligible plays screened: 1,788 (weeks 1-8).
Where chaining was possible (166 plays), windows chained 50% of the time, adding 0.19 s on average.
Sensitivity (either time, A vs B): 1.5 yd: 0.9 s vs 1.0 s; 2.0 yd: 0.6 s vs 1.0 s; 2.5 yd: 0.4 s vs 0.8 s


127.0.0.1 - - [10/Nov/2026 13:02:25] "GET /index.html HTTP/1.1" 200 -


**Limitations.** "In space" is distance to the nearest defender only. It ignores passing lanes, defender leverage and closing speed, pass rush pressure, and whether the quarterback was looking that way. The screen covers only plays with exactly two wide receivers running routes, so tight end and running back routes are not counted. The window is fixed at the last second before release, so plays with a quick throw and plays with a long hold are measured over the same interval. The metric describes the options the two receivers created, not where the ball went: in both selected plays the pass went to a tight end.